# Limpeza do dataset Spotify

Notebook separado para remover registros considerados ruídos ou fora do escopo da análise.

Critérios aplicados:
- Faixas com menos de 1 min
- Faixas sem BPM (0 BPM)
- Faixas com speechiness maior que 0.5
- Faixas dos gêneros de ruído/ambientais e clássicos (comedy, children, opera, gospel, piano, romance, classical, show-tunes, kids, ambient...)
- Faixas com baixa popularidade (<= 10)
- Faixas com baixa loudness (<= -16)
- Faixas com nome contendo "rain sounds", "white noise", etc.

In [1]:
# Instala pandas caso ainda não exista
try:
    import pandas as pd
except ModuleNotFoundError:
    !pip install pandas
    import pandas as pd

from pathlib import Path
from datetime import datetime

base_dir = Path.cwd()
csv_path = base_dir / 'dataset' / 'dataset.csv'
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_path = base_dir / 'dataset' / f'dataset_cleaned_{timestamp}.csv'

print(f'Arquivo de entrada: {csv_path}')
print(f'Arquivo de saída: {output_path}')


Arquivo de entrada: /home/gabe/re/r.ia-spotify_nano_challenge/dataset/dataset.csv
Arquivo de saída: /home/gabe/re/r.ia-spotify_nano_challenge/dataset/dataset_cleaned_20260831_110452.csv


In [2]:
df = pd.read_csv(csv_path)
print('Shape original:', df.shape)
print(df.head(3).to_string(index=False))
print('\nColunas:', list(df.columns))

Shape original: (114000, 21)
 Unnamed: 0               track_id                artists       album_name       track_name  popularity  duration_ms  explicit  danceability  energy  key  loudness  mode  speechiness  acousticness  instrumentalness  liveness  valence  tempo  time_signature track_genre
          0 5SuOikwiRyPMVoIQDJUgSV            Gen Hoshino           Comedy           Comedy          73       230666     False         0.676   0.461    1    -6.746     0       0.1430        0.0322          0.000001     0.358    0.715 87.917               4    acoustic
          1 4qPNDBW1i3p13qLCt0Ki3A           Ben Woodward Ghost (Acoustic) Ghost - Acoustic          55       149610     False         0.420   0.166    1   -17.235     1       0.0763        0.9240          0.000006     0.101    0.267 77.489               4    acoustic
          2 1iJBSr7s7jYXzM8EGcbK5b Ingrid Michaelson;ZAYN   To Begin Again   To Begin Again          57       210826     False         0.438   0.359    0    -9.734 

In [3]:
# Regras de remoção
undesirable_genres = {
    'comedy', 'children', 'opera', 'gospel', 'piano', 'romance',
    'classical', 'show-tunes', 'kids', 'ambient', 'new-age', 'soundtracks'
}

noise_keywords = [
    'rain sounds', 'white noise', 'nature sounds', 'sleep', 'meditation',
    'ambient', 'environmental sounds', 'soothing', 'soundscape'
]

# Normalização
df = df.copy()
df['track_genre'] = df['track_genre'].fillna('').astype(str).str.strip().str.lower()
df['track_name'] = df['track_name'].fillna('').astype(str).str.lower()

# Cria máscara de exclusão
mask = pd.Series(False, index=df.index)

mask |= df['duration_ms'] < 60000
mask |= df['tempo'] == 0
mask |= df['speechiness'] > 0.5
mask |= df['track_genre'].isin(undesirable_genres)
mask |= df['popularity'] <= 10
mask |= df['loudness'] <= -16

for keyword in noise_keywords:
    mask |= df['track_name'].str.contains(keyword, case=False, na=False)

# Totais por critério
criteria = {
    'duration_ms < 60000': df['duration_ms'] < 60000,
    'tempo == 0': df['tempo'] == 0,
    'speechiness > 0.5': df['speechiness'] > 0.5,
    'track_genre in blacklisted_genres': df['track_genre'].isin(undesirable_genres),
    'popularity <= 10': df['popularity'] <= 10,
    'loudness <= -16': df['loudness'] <= -16,
    'track_name contains noise keyword': False,
}

noise_mask = pd.Series(False, index=df.index)
for keyword in noise_keywords:
    noise_mask |= df['track_name'].str.contains(keyword, case=False, na=False)
criteria['track_name contains noise keyword'] = noise_mask

for name, cond in criteria.items():
    print(f'{name}: {int(cond.sum())} faixas removidas')

print(f'\nTotal de faixas removidas: {int(mask.sum())}')
print(f'Total restante: {int((~mask).sum())}')

duration_ms < 60000: 851 faixas removidas
tempo == 0: 157 faixas removidas
speechiness > 0.5: 1180 faixas removidas
track_genre in blacklisted_genres: 11000 faixas removidas
popularity <= 10: 23462 faixas removidas
loudness <= -16: 8108 faixas removidas
track_name contains noise keyword: 442 faixas removidas

Total de faixas removidas: 36272
Total restante: 77728


In [4]:
cleaned_df = df.loc[~mask].copy()
print('Shape final:', cleaned_df.shape)
print(cleaned_df.head(5).to_string(index=False))

Shape final: (77728, 21)
 Unnamed: 0               track_id                              artists                  album_name           track_name  popularity  duration_ms  explicit  danceability  energy  key  loudness  mode  speechiness  acousticness  instrumentalness  liveness  valence   tempo  time_signature track_genre
          0 5SuOikwiRyPMVoIQDJUgSV                          Gen Hoshino                      Comedy               comedy          73       230666     False         0.676   0.461    1    -6.746     0       0.1430        0.0322          0.000001    0.3580   0.7150  87.917               4    acoustic
          2 1iJBSr7s7jYXzM8EGcbK5b               Ingrid Michaelson;ZAYN              To Begin Again       to begin again          57       210826     False         0.438   0.359    0    -9.734     1       0.0557        0.2100          0.000000    0.1170   0.1200  76.332               4    acoustic
          4 5vjLSffimiIP26QG5WcN2K                     Chord Overstreet       

In [5]:
if output_path.exists():
    output_path.unlink()

cleaned_df.to_csv(output_path, index=False)
print(f'Arquivo salvo em: {output_path}')
print(f'Novo arquivo criado: {output_path.exists()}')


Arquivo salvo em: /home/gabe/re/r.ia-spotify_nano_challenge/dataset/dataset_cleaned_20260831_110452.csv
Novo arquivo criado: True


## Resultado da limpeza

Este notebook remove as faixas que se enquadram nos critérios definidos e salva um arquivo limpo em `dataset/dataset_cleaned.csv`.

Você pode usar esse dataset limpo em consultas SQL, análise exploratória ou treino de modelos.